Problem: 105. Construct Binary Tree from Preorder and Inorder Traversal

Category: Tree

Key Points: 
            1. first element in pre-order is root,   
            2. elements left of root in in-order are left subtree,   
            3. right of root are right subtree,   
            4. recursively build subtrees;  

            5. recursive algorithm: starts with basic cases: Null value

Method1: Divide and Conquer + Slicing


在二叉树重建问题中，代码通过以下三步来“分治”：

- 找根节点（分）：利用先序遍历（Preorder）的特性，preorder[0] 必然是当前树的根节点。

- 划分左右子树（治）：利用中序遍历（Inorder）的特性，在 inorder 中找到根节点的位置 mid。以此为分界线，mid 左边的元素全部属于左子树，右边的元素全部属于右子树。

- 递归切割（合）：使用 Python 的切片操作 preorder[1 : mid + 1] 和 inorder[:mid] 截取新的子数组，分别投喂给下一层的 buildTree 函数。

In [ ]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:

        # Base cases: Null value
        if not preorder or not inorder:
            return None

        # root value
        root = TreeNode(preorder[0])
        mid = inorder.index(preorder[0])

        # Create the left subtree
        root.left = self.buildTree(preorder[1: mid + 1], inorder[: mid])

        # Create the right subtree
        root.right = self.buildTree(preorder[mid + 1:], inorder[mid + 1: ])

        # Return the BST
        return root

Time Complexity: N 层递归×O(N) 的每层消耗=O(N^2)

- ```inorder.index()``` 是线性查找：树的节点并没有自带索引。要在 inorder 列表中找到根节点的位置，Python 必须从头到尾挨个检查（线性查找）。如果这棵树退化成一条链表，查找就需要执行 N 次、 N−1 次、 N−2 次……平均每次耗时 O(N)。

- 列表切片是内存复制： 切片 preorder[1: mid + 1]，在 Python 中，对列表进行切片（如 a[1:5]）不是创建引用，而是在内存中完整复制出一份新列表。复制操作的时间和空间复杂度也是 O(K)（K 为切片长度）。


Method2: Hash table + Two pointers

用空间换时间，用指针代替复制。  
trade space for time, and replace data duplication with pointers


- Eliminating the O(N) lookup of ```.index()```: Since there are no duplicate values in the inorder traversal, we can pre-populate a Hash Map (Dictionary) with all values and their corresponding indices before the algorithm even begins. Consequently, any subsequent lookup for the mid index drops to a constant time of O(1).

- Eliminating the overhead of ```array slicing```: Instead of copying data, we pass boundary pointers (left and right indices) to inform the next layer of recursion exactly which subarray segment it needs to process.

In [ ]:
class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:

    # HashMapping inorder's values
    inorder_map = {val: idx for idx, val in enumerate(inorder)}

    def helper(pre_left: int, pre_right: int, in_left: int, in_right: int) -> Optional[TreeNode]:
        
        # Base case: if the current subtree is empty
        if pre_left > pre_right or in_left > in_right:
            return None

        # The 1st elem in preorder is the root of the current subtree
        root_val = preorder[pre_left]
        root = TreeNode(root_val)

        # Use the hash map to find the index of the root in inorder
        mid = inorder_map[root_val]

        # Calculate the size of the left subtree
        left_size = mid - in_left

        # Recursively build the left and right subtrees
        root.left = helper(pre_left + 1, pre_left + left_size, in_left, mid - 1)
        root.right = helper(pre_left + left_size + 1, pre_right, mid + 1, in_right)

        return root

    # Start the recursion with the full range of preorder and inorder
    n = len(preorder)
    return helper(0, n - 1, 0, n - 1)
